# class_struct — Kaggle (Qwen 1.5B models)

Settings: **Internet on**. Start with Accelerator **None**, run setup, then switch to **GPU T4**.
Add-ons → Secrets → `HF_TOKEN` (Hugging Face write/read token).

Run **perturbation** for both Qwens first. Do not start crosslang until those CSVs are downloaded — Java/C# will OOM RAM if you leave `--languages` off.

In [ ]:
# CPU is fine for this cell. Do not enable GPU yet.
import os
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
except Exception as e:
    print("no kaggle secret HF_TOKEN:", e)

REPO = Path("/kaggle/working/mech-interp")
if not REPO.exists():
    !git clone -q -b main https://github.com/nolanlwin/code-model-interpretability.git {REPO}
%cd {REPO}
!git log -1 --oneline
!pip install -q -r pipeline/requirements.txt
!hf download dhyuti-n/xlcost-variable-roles --repo-type dataset --local-dir dataset
print("setup done — now set Accelerator to GPU T4 and run the next cells")

In [ ]:
# GPU T4. Model A — Python renaming sweep.
%cd /kaggle/working/mech-interp
!python -m pipeline.run_experiment perturbation --role class_struct \
    --model Qwen/Qwen2.5-1.5B --dataset dataset --split train
!ls -R results/unified/Qwen2.5-1.5B/class_struct/perturbation

In [ ]:
# GPU T4. Model B — Python renaming sweep.
%cd /kaggle/working/mech-interp
!python -m pipeline.run_experiment perturbation --role class_struct \
    --model Qwen/Qwen2.5-Coder-1.5B --dataset dataset --split train
!ls -R results/unified/Qwen2.5-Coder-1.5B/class_struct/perturbation

In [ ]:
# Download these before the session dies.
%cd /kaggle/working/mech-interp
!zip -r /kaggle/working/class_struct_qwen_perturbation.zip results/unified
print("Output → class_struct_qwen_perturbation.zip")

In [ ]:
# Crosslang later. Skip Java/C# on the first pass.
# Needs the --languages flag (this working copy of pipeline/run_experiment.py).
%cd /kaggle/working/mech-interp
!python -m pipeline.run_experiment crosslang --role class_struct \
    --model Qwen/Qwen2.5-1.5B --dataset dataset --split train \
    --languages C++ Javascript C
!python -m pipeline.run_experiment crosslang --role class_struct \
    --model Qwen/Qwen2.5-Coder-1.5B --dataset dataset --split train \
    --languages C++ Javascript C